# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD - Ablation Study)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3)  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub and `cd` into `ablation_study/`.
3. Link shared dataset (`data/final`) from `/content/drive/MyDrive/durian-ripeness-cmkd/`, and link `outputs/checkpoint/{teachermodal,studentmodal}` & `outputs/result/{results_teacher,results_student,test_student}` to `/content/drive/MyDrive/durian-ripeness-cmkd/ablation_study/`.
4. Launch 2-phase distillation training (`src/train_teacher_and_student/train_student.py`) and export the benchmark comparison table.

In [1]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

Mounted at /content/drive
✅ Google Drive mounted successfully!
Mon Nov  9 14:15:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |         

In [2]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = ""
REPO_DIR = "/content/durian-ripeness-classification"
ABLATION_DIR = "/content/durian-ripeness-classification/ablation_study"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
    os.makedirs(ABLATION_DIR, exist_ok=True)
    os.chdir(ABLATION_DIR)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "outputs"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in ablation_study root directory: {os.getcwd()}")

/content
Cloning into '/content/durian-ripeness-classification'...
remote: Enumerating objects: 533, done.
remote: Counting objects: 100% (533/533), done.
remote: Compressing objects: 100% (395/395), done.
remote: Total 533 (delta 211), reused 439 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (533/533), 24.30 MiB | 20.38 MiB/s, done.
Resolving deltas: 100% (211/211), done.
/content/durian-ripeness-classification
✅ Working in project root directory: /content/durian-ripeness-classification


In [3]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
REQ_PATH = "../requirements.txt" if os.path.isfile("../requirements.txt") else "requirements.txt"
!pip install -q uv
!uv pip install --system -r "$REQ_PATH"
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Resolved 167 packages in 858ms                                       
Prepared 13 packages in 1.38s                                            
Uninstalled 3 packages in 21ms
Installed 13 packages in 112ms                              
 + async-lru==2.4.0
 + comm==0.2.3
 - ipykernel==6.17.1
 + ipykernel==7.4.0
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.2.4
 - jupyter-client==7.4.9
 + jupyter-client==8.10.0
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.6
 + jupyterlab-server==2.28.1
 + nest-asyncio2==1.7.4
 - notebook==6.5.7
 + notebook==7.6.5
✅ Installed all dependencies successfully!


In [4]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_ABLATION_ROOT = DRIVE_PROJECT_ROOT / "ablation_study"
    DRIVE_UNIMODAL_CKPT = DRIVE_PROJECT_ROOT / "checkpoints" / "unimodal"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_TEACHER_CKPT = DRIVE_ABLATION_ROOT / "teachermodal"
    DRIVE_STUDENT_CKPT = DRIVE_ABLATION_ROOT / "studentmodal"
    DRIVE_RESULTS_TEACHER = DRIVE_ABLATION_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_ABLATION_ROOT / "results_student"
    DRIVE_TEST_STUDENT = DRIVE_ABLATION_ROOT / "test_student"

    # Pre-create ablation_study folders on Google Drive for real-time persistence
    DRIVE_TEACHER_CKPT.mkdir(parents=True, exist_ok=True)
    DRIVE_STUDENT_CKPT.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)
    DRIVE_TEST_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink shared unimodal/data to repo root (../) and Student/Teacher ablation outputs to Drive
    !rm -rf ../checkpoints/unimodal outputs/checkpoint/teachermodal outputs/checkpoint/studentmodal
    !rm -rf outputs/result/results_student outputs/result/test_student
    !mkdir -p ../checkpoints ../data/final outputs/checkpoint outputs/result
    !ln -sfn "{str(DRIVE_UNIMODAL_CKPT)}" ../checkpoints/unimodal
    !ln -sfn "{str(DRIVE_TEACHER_CKPT)}" outputs/checkpoint/teachermodal
    !ln -sfn "{str(DRIVE_STUDENT_CKPT)}" outputs/checkpoint/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" outputs/result/results_student
    !ln -sfn "{str(DRIVE_TEST_STUDENT)}" outputs/result/test_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf outputs/result/results_teacher
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" outputs/result/results_teacher
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf ../data/final/{split_name}
            !ln -sfn "{str(src_split)}" ../data/final/{split_name}

Path("outputs/checkpoint/studentmodal").mkdir(parents=True, exist_ok=True)
Path("outputs/result/results_student").mkdir(parents=True, exist_ok=True)
Path("outputs/result/test_student").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("outputs/checkpoint/teachermodal/best_teacher.pth").exists(), "->", Path("outputs/checkpoint/teachermodal").resolve())
print(" - Studentmodal   :", Path("outputs/checkpoint/studentmodal").resolve())
print(" - Results student:", Path("outputs/result/results_student").resolve())
print(" - Test student   :", Path("outputs/result/test_student").resolve())
print(" - Manifest CSV   :", Path("../data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [5]:
# ==============================================================================
# 5. RUN VISION-ONLY STUDENT DISTILLATION TRAINING (PHASE 2 - CMKD)
# ==============================================================================
!python src/train_teacher_and_student/train_student.py \
    --manifest ../data/final/final_multimodal_manifest.csv \
    --data-dir ../data/final \
    --teacher-ckpt outputs/checkpoint/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_tiny \
    --save-path outputs/checkpoint/studentmodal/best_student.pth \
    --results-dir outputs/result/results_student \
    --alpha 1.0 \
    --beta 2.0 \
    --gamma 1.0 \
    --temperature 3.0 \
    --feat-loss-type cosine \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 16 \
    --epochs 30 \
    --warmup-epochs 3 \
    --lr 5e-4 \
    --lr-backbone 1e-4 \
    --min-lr 1e-6 \
    --patience 10 \
    --select-metric fruit_macro_f1 \
    --eval-test \
    --device cuda

[2026-11-09 14:16:28] ==============================================================================
[2026-11-09 14:16:28] TRAINING VISION-ONLY STUDENT VIA CMKD (PHASE 2)
[2026-11-09 14:16:28] Device: cuda | AMP: True (torch.bfloat16) | batch 16 x accum 1
[2026-11-09 14:16:28] Student backbone : convnextv2_tiny.fcmae_ft_in1k
[2026-11-09 14:16:28] Distill weights  : alpha=1.0 (CE) | beta=2.0 (KD, T=3.0) | gamma=1.0 (Feat: cosine)
[2026-11-09 14:16:28] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/best_student.pth
[2026-11-09 14:16:28] ==============================================================================
[2026-11-09 14:16:28] Loaded Frozen Teacher from: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
/content/durian-ripeness-classification/src/datasets/sensor_dataset.py:132: UserWarning: Could not load /content/durian-ripeness-classification/workspaces/member3_sensor/02_train_unimodal_sensor/results/sensor_p

In [6]:
# ==============================================================================
# 6. INDEPENDENT RGB-ONLY EVALUATION & DISPLAY BENCHMARK COMPARISON TABLE
# ==============================================================================
!python src/train_teacher_and_student/eval_student.py \
    --checkpoint outputs/checkpoint/studentmodal/best_student.pth \
    --manifest ../data/final/final_multimodal_manifest.csv \
    --data-dir ../data/final \
    --split test \
    --output-dir outputs/result/results_student

comp_md = Path("outputs/result/results_student/benchmark_comparison.md")
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", comp_md.parent.resolve())

[2026-11-09 15:13:44] [INFO] [eval_teacher]: Initialized TeacherEvaluator on device: cuda (input_mode=vision)
[2026-11-09 15:13:44] [INFO] [eval_teacher]: Evaluation results will be saved to: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
[2026-11-09 15:13:44] [INFO] [eval_teacher]: === STARTING COMPREHENSIVE TEACHER MODEL EVALUATION ===
[2026-11-09 15:13:44] [INFO] [eval_teacher]: model=vision_student | split=test | input_mode=vision | strategy=mean_prob
[2026-11-09 15:13:44] [INFO] [eval_teacher]: Starting sample-level evaluation (mask_modality=None)...
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Inferred 298 sample, accuracy = 0.4966
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Aggregating fruit-level (fruit_id) predictions using strategy: mean_prob...
[2026-11-09 15:13:52] [INFO] [eval_teacher]: Aggregated 24 fruits, fruit-level accuracy = 0.5833
[2026-11-09 15:13:53] [INFO] [eval_teacher]: Starting Modality Abla